In [1]:
import pandas as pd 
import numpy as np 
import os 

In [2]:
metadata = pd.read_csv('../../data/metadata_demographic.csv')
metadata = metadata.drop_duplicates(subset='Participant_ID')
metadata = metadata.dropna(subset=['pd'])
metadata

,Protocol,Participant_ID,Task,gender,age,race,pd
0,SuperPD,NIHFT628PHTAY,ahhhh,female,70.0,White,no
1,SuperPD,NIHNT179KNNF4,ahhhh,female,70.0,White,yes
2,SuperPD,NIHYM875FLXFF,ahhhh,female,73.0,White,no
3,SuperPD,NIHBV117HUCTC,ahhhh,female,60.0,White,no
4,SuperPD,NIHZY217YWJA8,ahhhh,female,69.0,White,yes
...,...,...,...,...,...,...,...
1996,ValorPD,MOyJjyLX9hPvP3FSlLNYaG28xA23,NaN,M,78.0,White,yes
1997,ValorPD,zn6iI7uiq0U0xWTG5fuwR7yX9IH3,NaN,F,62.0,White,no
1998,ValorPD,XFJkSNMEpNgUnWg5Ouc6AWMOnQ82,NaN,F,62.0,White,yes
1999,ValorPD,j4GZI8ZFugZHRPxCKLBHC3DFwZE2,NaN,M,70.0,White,yes


In [3]:
df_pd_weigh_in = pd.read_csv("../../data/clinical_matched_finger_tapping_smile_qbf_with_features.csv")
df_pd_weigh_in = df_pd_weigh_in.drop_duplicates(subset='participant_id')
df_pd_weigh_in.shape

(304, 328)

In [4]:
metadata = metadata[metadata['Participant_ID'].isin(df_pd_weigh_in['participant_id'])]
metadata.shape

(303, 7)

In [5]:
def create_distribution_table_with_order(df, diagnosis_col, target_col, first_row_text, value_order=None):
    """
    Create a table showing counts and percentages for a specified column split by a diagnosis column.
    
    Parameters:
        df (pd.DataFrame): The input dataset.
        diagnosis_col (str): The column that indicates diagnosis (e.g., "Diagnosis").
        target_col (str): The column for which distribution is calculated (e.g., "Sex").
        first_row_text (str): The first row text for the table header (e.g., "Sex, n (%)").
        value_order (list): The preferred order of column values (e.g., ['Male', 'Female', ...]).
        
    Returns:
        pd.DataFrame: The distribution table.
    """
    # Calculate counts within each group (With PD and Without PD)
    with_pd_total = df[df[diagnosis_col] == 1].shape[0]
    without_pd_total = df[df[diagnosis_col] != 1].shape[0]

    # Counts for each target value in the groups
    with_pd_counts = df[df[diagnosis_col] == 1][target_col].value_counts()
    without_pd_counts = df[df[diagnosis_col] != 1][target_col].value_counts()
    total_counts = df[target_col].value_counts()

    # Use value_order or infer from the data
    if value_order is None:
        value_order = total_counts.index.tolist()

    # Create the rows for each unique value in the preferred order
    rows = []
    for value in value_order:
        # Get counts for each group
        with_pd_count = with_pd_counts.get(value, 0)
        without_pd_count = without_pd_counts.get(value, 0)
        total_count = total_counts.get(value, 0)

        # Calculate percentages within each group
        with_pd_pct = (with_pd_count / with_pd_total) * 100 if with_pd_total > 0 else 0
        without_pd_pct = (without_pd_count / without_pd_total) * 100 if without_pd_total > 0 else 0
        total_pct = (total_count / df.shape[0]) * 100

        # Add row to the table
        rows.append([
            first_row_text,
            value,
            f"{with_pd_count} ({with_pd_pct:.1f}%)",
            f"{without_pd_count} ({without_pd_pct:.1f}%)",
            f"{total_count} ({total_pct:.1f}%)"
        ])

    # Create the table DataFrame
    table = pd.DataFrame(rows, columns=["Demographic Property", "Attribute", "PD", "Control", "Total"])

    # # Add the first row text
    # first_row = pd.DataFrame([{
    #     "Demographic Property": first_row_text,
    #     "Attribute": "",
    #     "With PD": "",
    #     "Without PD": "",
    #     "Total": ""
    # }])

    # # Concatenate the header and the data rows
    # table = pd.concat([first_row, table], ignore_index=True)

    for index in range(1, len(table)):
        table.loc[index, 'Demographic Property'] = ''
    
    return table


In [6]:
metadata['Protocol'].value_counts(dropna=False)

Protocol
ValorPD        157
SuperPD_old     60
RoutePD         47
SuperPD         39
Name: count, dtype: int64

In [7]:
metadata['pd'].value_counts(dropna=False)

pd
no         161
yes         95
PD          37
Control     10
Name: count, dtype: int64

In [8]:
# Mapping the 'pd' column to binary values (0 and 1)
# Assuming 'no', 'Control', and 'Unlikely' are mapped to 0 (non-PD)
# and the rest are mapped to 1 (PD)

pd_map = {
    'no': 0,
    'Control': 0,
    'Unlikely': 0,
    'yes': 1,
    'Possible': 1,
    'PD': 1,
    'Probable': 1
}

metadata['Diagnosis'] = metadata['pd'].map(pd_map)
metadata['Diagnosis'].value_counts(dropna=False)

/tmp/ipykernel_2543756/3990877965.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['Diagnosis'] = metadata['pd'].map(pd_map)


Diagnosis
0    171
1    132
Name: count, dtype: int64

In [9]:
# Calculate counts and percentages for 'With PD' and 'Without PD'
with_pd_count = (metadata['Diagnosis'] == 1).sum()  # Assuming '1' indicates 'With PD'
without_pd_count = (metadata['Diagnosis'] != 1).sum()
total_count = with_pd_count + without_pd_count

with_pd_percentage = (with_pd_count / total_count) * 100
without_pd_percentage = (without_pd_count / total_count) * 100

# Create the final table structure
table_dict = {
    "Demographic Property": "",
    "Attribute": ["Number of Participants, n (%)"],
    "PD": [f"{with_pd_count} ({with_pd_percentage:.1f}%)"],
    "Control": [f"{without_pd_count} ({without_pd_percentage:.1f}%)"],
    "Total": [f"{total_count} (100%)"]
}

# Convert to a DataFrame for display
pd_count_table = pd.DataFrame(table_dict)

pd_count_table

,Demographic Property,Attribute,PD,Control,Total
0,,"Number of Participants, n (%)",132 (43.6%),171 (56.4%),303 (100%)


In [10]:
metadata['gender'].value_counts()

gender
F         85
male      76
M         72
female    70
Name: count, dtype: int64

In [11]:
gender_map = {
    "female": "Female",
    "Female": "Female",
    'F': 'Female',
    "male": "Male",
    "Male": "Male",
    'M': 'Male',
    "Prefer not to respond": "Unknown",
    "Nonbinary": "Non-Binary"
}
metadata['gender_normalized'] = metadata['gender'].map(gender_map)
metadata['gender_normalized'].value_counts(dropna=False)

/tmp/ipykernel_2543756/2488056917.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['gender_normalized'] = metadata['gender'].map(gender_map)


gender_normalized
Female    155
Male      148
Name: count, dtype: int64

In [12]:
# Define the preferred order
preferred_order = ['Female', 'Male',]

# Call the function with the preferred order
sex_table_with_order = create_distribution_table_with_order(
    df=metadata,
    diagnosis_col="Diagnosis",
    target_col="gender_normalized",
    first_row_text="Sex",
    value_order=preferred_order
)


sex_table_with_order

,Demographic Property,Attribute,PD,Control,Total
0,Sex,Female,59 (44.7%),96 (56.1%),155 (51.2%)
1,,Male,73 (55.3%),75 (43.9%),148 (48.8%)


In [13]:
metadata['age'].value_counts(dropna=False)

age
66.000000    13
70.000000    10
64.000000    10
65.000000     9
71.000000     9
             ..
75.169237     1
78.816129     1
72.365620     1
75.864551     1
45.000000     1
Name: count, Length: 103, dtype: int64

In [14]:
metadata['age'] = metadata['age'].apply(lambda x: np.nan if x < 15 or x > 100 else x)
# Display the value counts of the 'age' column sorted by ascending order of the age values
metadata['age'].value_counts(dropna=False).sort_index()


/tmp/ipykernel_2543756/2598866290.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['age'] = metadata['age'].apply(lambda x: np.nan if x < 15 or x > 100 else x)


age
22.000000    1
24.000000    1
25.000000    1
26.000000    1
27.000000    3
            ..
79.577267    1
81.496654    1
86.000000    1
87.000000    2
NaN          3
Name: count, Length: 102, dtype: int64

In [15]:
# Initialize 'age_normalized' column with NaN
metadata['age_normalized'] = np.nan

# Ensure 'age' is numeric where possible
def safe_numeric(x):
    try:
        return float(x)
    except (ValueError, TypeError):
        return np.nan

metadata['age_numeric'] = metadata['age'].apply(safe_numeric)

# Define conditions for numeric age ranges
conditions = [
    metadata['age_numeric'] < 20,
    (metadata['age_numeric'] >= 20) & (metadata['age_numeric'] <= 39),
    (metadata['age_numeric'] >= 40) & (metadata['age_numeric'] <= 59),
    (metadata['age_numeric'] >= 60) & (metadata['age_numeric'] <= 79),
    metadata['age_numeric'] >= 80
]

# Define labels for the conditions
age_labels = [
    '< 20',
    '20 - 39',
    '40 - 59',
    '60 - 79',
    '>= 80'
]

# Apply conditions to normalize 'age'
metadata['age_normalized'] = np.select(
    conditions,
    age_labels,
    default='Not Mentioned'
)


/tmp/ipykernel_2543756/3161752875.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['age_normalized'] = np.nan
/tmp/ipykernel_2543756/3161752875.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['age_numeric'] = metadata['age'].apply(safe_numeric)
/tmp/ipykernel_2543756/3161752875.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://panda

In [16]:
import numpy as np

def process_age(age):
    try:
        # If the age is already numeric, return it
        if isinstance(age, (int, float)):
            return float(age)
        # If the age is a range like "50-60", calculate the mean
        if isinstance(age, str) and '-' in age:
            start, end = map(float, age.split('-'))
            return (start + end) / 2
    except:
        pass
    # Return NaN for invalid entries
    return np.nan

# Apply the processing function to handle all cases
metadata['age_processed'] = metadata['age'].apply(process_age)

# Calculate the mean and range (ignoring NaN values)
mean_age = metadata['age_processed'].mean()
min_age = metadata['age_processed'].min()
max_age = metadata['age_processed'].max()

# Print the results
print(f"Mean age: {mean_age:.2f}")
print(f"Age range: {min_age:.2f} - {max_age:.2f}")


Mean age: 60.60
Age range: 22.00 - 87.00


/tmp/ipykernel_2543756/563915863.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['age_processed'] = metadata['age'].apply(process_age)


In [17]:
metadata['age_normalized'].value_counts(dropna=False)

age_normalized
60 - 79          186
40 - 59           70
20 - 39           37
Not Mentioned      6
>= 80              4
Name: count, dtype: int64

In [18]:
# Define the preferred order
preferred_order = [
    '< 20',
    '20 - 39',
    '40 - 59',
    '60 - 79',
    '>= 80',
    'Not Mentioned'
]



# Call the function with the preferred order
age_table_with_order = create_distribution_table_with_order(
    df=metadata,
    diagnosis_col="Diagnosis",
    target_col="age_normalized",
    first_row_text=f"Age in years (range: {min_age:.1f} - {max_age:.1f}, mean: {mean_age:.1f}), n (%)",
    value_order=preferred_order
)


age_table_with_order

,Demographic Property,Attribute,PD,Control,Total
0,"Age in years (range: 22.0 - 87.0, mean: 60.6),...",< 20,0 (0.0%),0 (0.0%),0 (0.0%)
1,,20 - 39,0 (0.0%),37 (21.6%),37 (12.2%)
2,,40 - 59,21 (15.9%),49 (28.7%),70 (23.1%)
3,,60 - 79,104 (78.8%),82 (48.0%),186 (61.4%)
4,,>= 80,3 (2.3%),1 (0.6%),4 (1.3%)
5,,Not Mentioned,4 (3.0%),2 (1.2%),6 (2.0%)


In [19]:
metadata['race'].value_counts(dropna=False)

race
White                               246
['White']                            43
Prefer Not to Answer                  7
['Black or African American']         3
Black or African American             1
American Indian or Alaska Native      1
['Prefer not to answer']              1
Asian                                 1
Name: count, dtype: int64

In [20]:
def map_race_simplified(value):
    if isinstance(value, str):
        value = value.lower()  # Make case-insensitive
        if 'asian' in value:
            return "Asian"
        elif 'nativeamerican' in value or 'american indian' in value:
            return "American Indian or Alaska Native"
        elif 'black' in value:
            return "Black or African American"
        elif 'nativepacific' in value or 'hawaiian' in value:
            return "Native Hawaiian or Other Pacific Islander"
        elif 'white' in value:
            return "White"
        elif 'other' in value or 'on' in value:
            return "Others"
        elif 'prefer not to' in value or '[]' in value:
            return "Not Mentioned"
    return "Not Mentioned"  # Default to Unknown if value is not a string or doesn't match

# Apply the function to the 'race' column
metadata['race_normalized'] = metadata['race'].apply(map_race_simplified)

# Display the cleaned race column counts
print(metadata['race_normalized'].value_counts(dropna=False))

race_normalized
White                               289
Not Mentioned                         8
Black or African American             4
American Indian or Alaska Native      1
Asian                                 1
Name: count, dtype: int64


/tmp/ipykernel_2543756/2427306195.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['race_normalized'] = metadata['race'].apply(map_race_simplified)


In [21]:
# Define the preferred order
preferred_order = [
    "White",
    "Asian",
    "Black or African American",
    "American Indian or Alaska Native",
    "Others",
    "Not Mentioned"
]

# Call the function with the preferred order
race_table_with_order = create_distribution_table_with_order(
    df=metadata,
    diagnosis_col="Diagnosis",
    target_col="race_normalized",
    first_row_text="Race, n (%)",
    value_order=preferred_order
)


race_table_with_order

,Demographic Property,Attribute,PD,Control,Total
0,"Race, n (%)",White,128 (97.0%),161 (94.2%),289 (95.4%)
1,,Asian,1 (0.8%),0 (0.0%),1 (0.3%)
2,,Black or African American,2 (1.5%),2 (1.2%),4 (1.3%)
3,,American Indian or Alaska Native,1 (0.8%),0 (0.0%),1 (0.3%)
4,,Others,0 (0.0%),0 (0.0%),0 (0.0%)
5,,Not Mentioned,0 (0.0%),8 (4.7%),8 (2.6%)


In [22]:
metadata['Protocol'].value_counts(dropna=False)

Protocol
ValorPD        157
SuperPD_old     60
RoutePD         47
SuperPD         39
Name: count, dtype: int64

In [23]:
metadata[metadata['Diagnosis'] == 1]['Protocol'].value_counts(dropna=False)

Protocol
SuperPD_old    37
RoutePD        37
ValorPD        31
SuperPD        27
Name: count, dtype: int64

In [24]:
env_map = {
    'ParkTest':'Home-Global',
    'ValorPD': 'Clinic',
    'InMotion': 'PD Care Facility',
    'ValidationStudy': 'Clinic',
    'SuperPD': 'Clinic',
    'ClusterPD': 'Clinic',
    'SuperPD_old': 'Clinic',
    'SuperPD': 'Clinic',
    'RoutePD': 'PD Care Facility'
    
}

metadata['env'] = metadata['Protocol'].map(env_map)
metadata['env'].value_counts(dropna=False)

/tmp/ipykernel_2543756/1323468674.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['env'] = metadata['Protocol'].map(env_map)


env
Clinic              256
PD Care Facility     47
Name: count, dtype: int64

In [25]:
# Define the preferred order
preferred_order = [
    "Home-Global",
    "PD Care Facility",
    "Clinic",
]

# Call the function with the preferred order
env_table_with_order = create_distribution_table_with_order(
    df=metadata,
    diagnosis_col="Diagnosis",
    target_col="env",
    first_row_text="Recording Environment, n (%)",
    value_order=preferred_order
)
env_table_with_order

,Demographic Property,Attribute,PD,Control,Total
0,"Recording Environment, n (%)",Home-Global,0 (0.0%),0 (0.0%),0 (0.0%)
1,,PD Care Facility,37 (28.0%),10 (5.8%),47 (15.5%)
2,,Clinic,95 (72.0%),161 (94.2%),256 (84.5%)


In [26]:
total_table = pd.concat([
    pd_count_table, 
    sex_table_with_order, 
    age_table_with_order, 
    race_table_with_order, 
    env_table_with_order
], ignore_index=True)
total_table

,Demographic Property,Attribute,PD,Control,Total
0,,"Number of Participants, n (%)",132 (43.6%),171 (56.4%),303 (100%)
1,Sex,Female,59 (44.7%),96 (56.1%),155 (51.2%)
2,,Male,73 (55.3%),75 (43.9%),148 (48.8%)
3,"Age in years (range: 22.0 - 87.0, mean: 60.6),...",< 20,0 (0.0%),0 (0.0%),0 (0.0%)
4,,20 - 39,0 (0.0%),37 (21.6%),37 (12.2%)
5,,40 - 59,21 (15.9%),49 (28.7%),70 (23.1%)
6,,60 - 79,104 (78.8%),82 (48.0%),186 (61.4%)
7,,>= 80,3 (2.3%),1 (0.6%),4 (1.3%)
8,,Not Mentioned,4 (3.0%),2 (1.2%),6 (2.0%)
9,"Race, n (%)",White,128 (97.0%),161 (94.2%),289 (95.4%)


In [27]:
total_table.to_csv('../../data/demographic_table_pd_weigh_in_cohort.csv', index=False)  